# HW-05: Clustering with OWID Energy Data

In this notebook, you will:

- Load and explore the OWID energy dataset (`owid-energy.csv`)
- Clean and filter the data to a chosen time window
- Select and standardize numeric features for clustering
- Run K-Means clustering and evaluate choices of K using the Calinski–Harabasz score
- Run hierarchical (agglomerative) clustering
- Fit a Gaussian Mixture Model (GMM), assign cluster labels, and identify outliers
- Compute silhouette scores to evaluate clustering quality
- Interpret the resulting clusters in terms of energy profiles and decarbonization

> **Important:** All core logic must be implemented in `ds.py`. This notebook should only call functions from `ds.py` and use them for EDA, visualization, and written interpretation.

## 1. Setup

Import required libraries and your `ds` module.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import ds

plt.style.use("default")
%config InlineBackend.figure_format = "retina"

## 2. Load the OWID Energy Dataset

Use your `load_energy_data` and `initial_summary` functions to inspect the structure of `owid-energy.csv`.

In [ ]:
data_path = "data/owid-energy.csv"
df = ds.load_energy_data(data_path)
df.head()

In [ ]:
summary = ds.initial_summary(df)
summary

In the cell below, briefly describe:

- How many rows and columns the dataset has
- Which columns appear numeric vs. categorical
- Any immediate data quality issues you notice (e.g., many missing values in certain columns)

_Write your observations here._

## 3. Clean and Filter the Data

We will:

1. Clean the dataset using `clean_energy_data`.
2. Filter to a more recent time window using `filter_year_range` (for example, 2000–2022).

You may adjust the year range as needed, but be sure to explain your choice in a markdown cell.

In [ ]:
# Choose some key columns that should not be missing
subset_cols = [
    "fossil_energy_per_capita",
    "renewables_energy_per_capita",
    "low_carbon_share_energy"
]

df_clean = ds.clean_energy_data(df, subset_cols=subset_cols)
df_clean.shape

In [ ]:
# Filter to a recent time window (adjust if needed)
df_recent = ds.filter_year_range(df_clean, start_year=2000, end_year=2022)
df_recent[["country", "year"] + subset_cols].head()

_In a sentence or two, justify your choice of year range. Are you focusing on the modern energy transition? Avoiding early years with sparse data?_

## 4. Select and Standardize Features

Next, choose a set of numeric features for clustering. For example:

- `fossil_energy_per_capita`
- `renewables_energy_per_capita`
- `low_carbon_share_energy`

Then:
- Use `select_clustering_features` to create a feature-only DataFrame
- Use `standardize_features` to standardize them to mean 0 and std 1

In [ ]:
feature_cols = [
    "fossil_energy_per_capita",
    "renewables_energy_per_capita",
    "low_carbon_share_energy"
]

X_features = ds.select_clustering_features(df_recent, feature_cols)
X_features.head()

In [ ]:
X_scaled, means, stds = ds.standardize_features(X_features)
X_scaled.describe().T

Briefly comment on whether the standardized features look reasonable (means near 0, standard deviations near 1).

_Write your comments here._

## 5. K-Means: Choosing a Number of Clusters (K)

We will use the **Calinski–Harabasz score** to compare different choices of K.

Steps:
1. Use `compute_calinski_harabasz_scores` for a range of K (e.g., 2 to 8).
2. Plot K vs. score.
3. Choose a candidate K based on the score and your judgment.

In [ ]:
cluster_range = range(2, 9)
ch_scores = ds.compute_calinski_harabasz_scores(X_scaled, cluster_range)
ch_scores

In [ ]:
ks = sorted(ch_scores.keys())
scores = [ch_scores[k] for k in ks]

plt.figure()
plt.plot(ks, scores, marker="o")
plt.xlabel("Number of clusters (K)")
plt.ylabel("Calinski–Harabasz score")
plt.title("CH score vs. K (K-Means)")
plt.show()

_Based on this plot, what value of K seems reasonable? Explain your choice._

## 6. K-Means: Final Clustering and Cluster Profiles

Now run K-Means with your chosen K using `run_kmeans`.

Then:
- Attach the cluster labels back to the original `df_recent`
- Compute average feature values by cluster
- Optionally, examine clusters by country or region over time

In [ ]:
# Replace this with your chosen K (e.g., 3 or 4)
k_final = 3
labels_kmeans, centers_kmeans = ds.run_kmeans(X_scaled, n_clusters=k_final, random_state=42)
labels_kmeans[:10]

In [ ]:
df_kmeans = df_recent.copy()
df_kmeans["cluster_kmeans"] = labels_kmeans

cluster_profiles = df_kmeans.groupby("cluster_kmeans")[feature_cols].mean()
cluster_profiles

_How would you describe each K-Means cluster in terms of energy use and decarbonization?
For example: "Cluster 0: high fossil, low renewables" vs. "Cluster 1: low fossil, high low-carbon share"._

## 7. Hierarchical Clustering

Use `run_hierarchical_clustering` to perform agglomerative clustering on the same standardized features.

You can:
- Use the same **K** as for K-Means
- Compare labels and cluster profiles
- (Optional) Plot a dendrogram in a separate cell (using SciPy) — this is not required for grading, but can be interesting.

In [ ]:
labels_hier = ds.run_hierarchical_clustering(X_scaled, n_clusters=k_final, linkage="ward")
df_hier = df_recent.copy()
df_hier["cluster_hier"] = labels_hier

df_hier.groupby("cluster_hier")[feature_cols].mean()

_Compare K-Means and hierarchical clustering:
- Do they suggest similar groupings?
- Are certain countries consistently grouped together?_

## 8. Gaussian Mixture Models (GMM) and Outlier Detection

Now we will:

1. Fit a GMM using `fit_gmm` with `n_components = k_final`.
2. Get cluster labels using `gmm_cluster_labels`.
3. Detect potential outliers using `gmm_outlier_mask` with a probability threshold (e.g., 0.05).

Outliers are points with **low maximum cluster membership probability** under the GMM.

In [ ]:
gmm_model = ds.fit_gmm(X_scaled, n_components=k_final, random_state=42)
labels_gmm = ds.gmm_cluster_labels(gmm_model, X_scaled)
labels_gmm[:10]

In [ ]:
outlier_mask = ds.gmm_outlier_mask(gmm_model, X_scaled, threshold=0.05)
df_gmm = df_recent.copy()
df_gmm["cluster_gmm"] = labels_gmm
df_gmm["is_outlier"] = outlier_mask

df_gmm["is_outlier"].value_counts()

Inspect the most extreme outliers (if any):

In [ ]:
df_outliers = df_gmm[df_gmm["is_outlier"]]
df_outliers[["country", "year"] + feature_cols].head(10)

_Are the outliers meaningful? For example, are they very high fossil users, very high renewables users, or small countries with unusual energy profiles?_

## 9. Cluster Validation with Silhouette Score

We will use the silhouette score to compare different clustering approaches:

- K-Means labels
- Hierarchical clustering labels
- GMM labels (treating GMM as a hard clustering)

Use `silhouette_for_labels` on each set of labels.

In [ ]:
sil_kmeans = ds.silhouette_for_labels(X_scaled, labels_kmeans)
sil_hier = ds.silhouette_for_labels(X_scaled, labels_hier)
sil_gmm = ds.silhouette_for_labels(X_scaled, labels_gmm)

{
    "kmeans": sil_kmeans,
    "hierarchical": sil_hier,
    "gmm": sil_gmm,
}

_Which clustering method has the highest silhouette score? Does that match what you see when you look at the cluster profiles?_

## 10. Interpretation and Reflection

In your own words, briefly address the following:

1. **Cluster meanings:** What types of energy profiles emerge from your clustering (e.g., high fossil vs. high renewable vs. transitional)?
2. **Temporal patterns:** Do certain clusters appear more in recent years or earlier years?
3. **Geographical patterns:** Are particular regions (continents or income groups, if available) concentrated in specific clusters?
4. **Method comparison:** How do K-Means, hierarchical clustering, and GMM differ in the clusters they find? Do they tell a consistent story?
5. **Ethical and policy considerations:** What are some risks of using unsupervised clustering to label countries' energy systems (e.g., oversimplification, political misinterpretation, ignoring within-country inequalities)?

Write your reflection in the markdown cell below.

_Write your interpretation and reflection here._